# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIRˆ² dataset using the `mlcroissant` library. Following the Croissant schema, entities such as record sets, fields, and columns are referenced by their `@id`. 

### Dataset Source
The dataset is described by a Croissant schema at this URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import warnings
warnings.filterwarnings("ignore")

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
meta = ds.metadata

print(f"Dataset name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"License: {meta.license}")


## 2. Data Overview
Review the available record sets, fields, and their `@id`s. In Croissant, each record set and field is referenced by its `@id`, which is used consistently in every operation.

In [ ]:
# List available record sets and their fields
record_sets = []
print("Record sets (@id, name, description):\n")
for rs in meta.record_sets:
    record_sets.append(rs['@id'])
    print(f"- @id: {rs['@id']}")
    print(f"  Name: {rs.get('name')}")
    print(f"  Description: {rs.get('description')}")
    print(f"  Fields:")
    for field in rs.get('fields', []):
        print(f"    - @id: {field['@id']} (name: {field.get('name')}, dataType: {field.get('dataType')})")
    print()

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame. Use the record set and field `@id`s from the above overview.

**Example:** We'll extract all records from each available record set for exploration.

In [ ]:
# Extract data from available record sets
dataframes = {}

for rs_id in record_sets:
    print(f"Loading records from record set: {rs_id}")
    records = list(ds.records(record_set=rs_id))
    if len(records) == 0:
        print("  No records found.\n")
        continue
    # Store as DataFrame
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Example rows:\n{df.head(2)}\n")
print(f"Loaded record sets: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps. For demonstration, we will:

- Select the main protein-abundance table (record set with protein records, e.g. `/protein_table`),
- Filter records for abundant proteins (`abundance > 10`: using the field `@id`),
- Normalize the abundance column,
- Group by another field such as a 'sample' or 'gene name' (depending on available fields in the dataset),
- Show example results.

**All fields are referenced by their `@id`. Please adjust the field IDs according to the actual table above (replace as needed for your dataset).**


In [ ]:
# Edit these to match the IDs from your record set and field overview above.
# For illustration, we use the first available record set and plausible numeric/categorical field ids.
if dataframes:
    chosen_record_set = list(dataframes.keys())[0]  # adjust if you want a different set
    df = dataframes[chosen_record_set]

    # Infer numeric and group field IDs by inspecting column names
    print(f"Fields for {chosen_record_set}: {df.columns.tolist()}")
    
    # Try to choose a numeric and a group field
    # Replace with the actual @id of your numeric field (e.g. 'abundance'), and group field (e.g. 'gene_name')
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    possible_group_fields = [col for col in df.columns if df[col].dtype == object]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]  # pick the first numeric field
    else:
        print("No obvious numeric field found.")
        numeric_field = df.columns[0]
    
    if possible_group_fields:
        group_field = possible_group_fields[0]  # pick first object field
    else:
        print("No obvious group field found.")
        group_field = df.columns[0]

    print(f"Using numeric field '@id': {numeric_field}")
    print(f"Using group field '@id': {group_field}\n")

    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize the distribution of the normalized numeric field and compare means across groups.

This example assumes you're using Matplotlib and Seaborn. If no numeric field was found, this cell will be skipped.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field in filtered_df.columns:
    plt.figure(figsize=(10, 4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field in filtered_df.columns:
        plt.figure(figsize=(10, 4))
        top_n = 12
        mean_per_group = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:top_n]
        mean_per_group.plot(kind='bar')
        plt.title(f"Top {top_n} {group_field} by mean {numeric_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print("Not enough data to plot.")

## 6. Conclusion

This notebook demonstrated loading and exploring a Croissant-formatted biomedical dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their `@id` for transparency. You can now extend this exploration with additional visualizations, statistical analysis, or machine learning pipelines using your selected fields.

**Key Observations:**
- Multiple record sets are available, each describing a different aspect of the experimental results.
- Data fields (columns) are referenced by their `@id` for reproducibility across scripts and pipelines.
- Common EDA and filtering steps can be performed directly from the Croissant schema and its record sets.

**Next steps:**
- Try identifying and analyzing different record sets and fields of interest.
- Join data across record sets when appropriate (using shared field `@id`s) for richer analysis.
- Use this pipeline as a reproducible and FAIR data science workflow template.
